In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../../../data/interim/punjab_cleaned_groundwater.csv')

C:\Users\raj vardhan jha\AppData\Local\Temp\ipykernel_12708\2700469795.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../../data/interim/punjab_cleaned_groundwater.csv')


In [3]:
df.shape

(681295, 9)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 681295 entries, 0 to 681294
Data columns (total 9 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   Station                                          681295 non-null  object 
 1   State                                            681295 non-null  object 
 2   District LGD Code                                681295 non-null  int64  
 3   District                                         681295 non-null  object 
 4   Block                                            515117 non-null  object 
 5   Latitude                                         681295 non-null  float64
 6   Longitude                                        681295 non-null  float64
 7   Data Acquisition Time                            681295 non-null  object 
 8   Groundwater Level Telemetry Quadridaily (meter)  681295 non-null  float64
dtypes: float64(3), i

In [5]:
# Date-Time Conversion: 

df["Data Acquisition Time"] = pd.to_datetime(df["Data Acquisition Time"]) 
df.dtypes

Station                                                    object
State                                                      object
District LGD Code                                           int64
District                                                   object
Block                                                      object
Latitude                                                  float64
Longitude                                                 float64
Data Acquisition Time                              datetime64[ns]
Groundwater Level Telemetry Quadridaily (meter)           float64
dtype: object

In [8]:
#F-1: Calendar-Based Features: 

df["Year"] = df["Data Acquisition Time"].dt.year
df["Month"] = df["Data Acquisition Time"].dt.month
df["Day"] = df["Data Acquisition Time"].dt.day
df["Day_of_Week"] = df["Data Acquisition Time"].dt.dayofweek
df["Week_of_Year"] = df["Data Acquisition Time"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Data Acquisition Time"].dt.quarter
df['Is_Weekend'] = (df['Day_of_Week'] >= 5).astype(int) 

In [9]:
df[
    [
        "Data Acquisition Time",
        "Year",
        "Month",
        "Day",
        "Day_of_Week",
        "Week_of_Year",
        "Quarter",
        "Is_Weekend",
    ]
].head()

,Data Acquisition Time,Year,Month,Day,Day_of_Week,Week_of_Year,Quarter,Is_Weekend
0,2020-03-20 13:31:00,2020,3,20,4,12,1,0
1,2020-03-20 13:32:00,2020,3,20,4,12,1,0
2,2020-03-20 18:00:00,2020,3,20,4,12,1,0
3,2020-03-21 06:00:00,2020,3,21,5,12,1,1
4,2020-03-21 12:00:00,2020,3,21,5,12,1,1


In [11]:
#F-2: Season Feature: 

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

df["Season"] = df["Month"].apply(get_season)
df['Season']

0         Summer
1         Summer
2         Summer
3         Summer
4         Summer
           ...  
681290    Winter
681291    Winter
681292    Winter
681293    Winter
681294    Winter
Name: Season, Length: 681295, dtype: object

In [12]:
# now since the data contains measurements from multiple stations. If we create lag features directly, the previous row for one station could come from a completely different station, which would produce incorrect values 
#so we will sort the data on the basis of 'Station' and 'Data Acquisition Time' 

df = df.sort_values(
    by=['Station', 'Data Acquisition Time']
).reset_index(drop=True)

In [13]:
df[['Station', 'Data Acquisition Time']].head(20)

,Station,Data Acquisition Time
0,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-20 13:31:00
1,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-20 13:32:00
2,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-20 18:00:00
3,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-21 06:00:00
4,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-21 12:00:00
5,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-21 18:00:00
6,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-22 00:00:00
7,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-22 06:00:00
8,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-22 12:00:00
9,AMRITSAR;AJNALA;AJNALA;CHAK DOGRA;Vet. Hospital,2020-03-22 18:00:00


In [14]:
#F-3: Lag Features: 
# Groundwater levels are highly autocorrelated, so previous values are often the strongest predictors of future values 

In [15]:
#Lag-1: 

df['GW_Lag_1'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .shift(1)
)
df['GW_Lag_1']

0            NaN
1         -8.675
2         -8.673
3         -8.557
4         -8.476
           ...  
681290   -25.000
681291   -25.000
681292   -25.000
681293   -25.000
681294   -25.000
Name: GW_Lag_1, Length: 681295, dtype: float64

In [16]:
#Lag-2: 

df['GW_Lag_2'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .shift(2)
) 
df['GW_Lag_2']

0            NaN
1            NaN
2         -8.675
3         -8.673
4         -8.557
           ...  
681290   -25.000
681291   -25.000
681292   -25.000
681293   -25.000
681294   -25.000
Name: GW_Lag_2, Length: 681295, dtype: float64

In [17]:
#Lag-3: 

df['GW_Lag_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .shift(3)
) 
df['GW_Lag_3']

0            NaN
1            NaN
2            NaN
3         -8.675
4         -8.673
           ...  
681290   -25.000
681291   -25.000
681292   -25.000
681293   -25.000
681294   -25.000
Name: GW_Lag_3, Length: 681295, dtype: float64

In [18]:
#F-4: Rolling Statistics Features: 

In [19]:
# Rolling Mean (Window = 3)

df['GW_Rolling_Mean_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .transform(lambda x: x.rolling(window=3).mean())
)
df['GW_Rolling_Mean_3']

0               NaN
1               NaN
2         -8.635000
3         -8.568667
4         -8.344333
            ...    
681290   -25.000000
681291   -25.000000
681292   -25.000000
681293   -25.000000
681294   -25.000000
Name: GW_Rolling_Mean_3, Length: 681295, dtype: float64

In [22]:
# Rolling Standard Deviation (Window = 3) 

df['GW_Rolling_STD_3'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .transform(lambda x: x.rolling(window=3).std())
)
df['GW_Rolling_STD_3']

0              NaN
1              NaN
2         0.067557
3         0.099017
4         0.300939
            ...   
681290    0.000000
681291    0.000000
681292    0.000000
681293    0.000000
681294    0.000000
Name: GW_Rolling_STD_3, Length: 681295, dtype: float64

In [25]:
df['GW_Expanding_Mean'] = (
    df.groupby('Station')['Groundwater Level Telemetry Quadridaily (meter)']
      .transform(lambda x: x.expanding().mean())
)
df['GW_Expanding_Mean']

0         -8.675000
1         -8.674000
2         -8.635000
3         -8.595250
4         -8.476200
            ...    
681290   -25.039691
681291   -25.039656
681292   -25.039621
681293   -25.039587
681294   -25.039552
Name: GW_Expanding_Mean, Length: 681295, dtype: float64

This feature answers:

"What has been the average groundwater level at this station up to the current observation?"

Unlike a rolling mean, which only looks at the last few observations, the expanding mean summarizes the station's historical behavior.

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 681295 entries, 0 to 681294
Data columns (total 24 columns):
 #   Column                                           Non-Null Count   Dtype         
---  ------                                           --------------   -----         
 0   Station                                          681295 non-null  object        
 1   State                                            681295 non-null  object        
 2   District LGD Code                                681295 non-null  int64         
 3   District                                         681295 non-null  object        
 4   Block                                            515117 non-null  object        
 5   Latitude                                         681295 non-null  float64       
 6   Longitude                                        681295 non-null  float64       
 7   Data Acquisition Time                            681295 non-null  datetime64[ns]
 8   Groundwater Level Teleme

In [27]:
df.to_csv('../../../data/processed/punjab_featured_groundwater.csv')